# NIST StRD: linear regression

Does Julia's linear regression give NIST's certified coefficients, standard errors, residual standard deviation and R-squared? The 11 linear regression datasets of NIST's Statistical Reference Datasets, among them Longley's and Filip's degree-10 polynomial, fitted with GLM's `lm`, which uses a Cholesky decomposition, and by QR.

The Statistical Reference Datasets (StRD) of the US National Institute of Standards and Technology (NIST) were built to test the numerical accuracy of statistical software. Each comes with values certified to 15 significant digits, computed in multiple precision. Some are real data; others are constructed to be hard, with many leading digits that do not change from one observation to the next, so that software which loses precision gives wrong digits. NIST grades each dataset's difficulty as lower, average or higher.

The data files sit beside this notebook as NIST publishes them (https://www.itl.nist.gov/div898/strd/), and every certified value is read from its file. The number of significant digits a value shares with NIST is the log relative error of McCullough (1998, *The American Statistician* 52(4), 358-366), the usual measure for these datasets: -log10(|value - certified| / |certified|), or -log10|value| when the certified value is 0, rounded down and at most 15.

**The data limit.** Julia stores each number of a file as a Float64, a binary number with 15 to 17 significant digits, and storing alone can change a number: `1000000000000.4` is stored as `1000000000000.4000244140625`. The **data limit** repeats each calculation in 256-bit `BigFloat` arithmetic on the stored numbers: no Float64 program can do better on this data. Where Julia falls short of the data limit, its method lost digits; where the data limit falls short of 15, the storing did.

In [ ]:
Pollis.packages("GLM")

## Set-up

`read_strd` reads a NIST file, `certified` reads a certified value from its header, `digits_same` counts the significant digits a value shares with it, `report` prints a dataset's values below NIST's and `summary` the fewest digits in each dataset.

In [ ]:
using GLM, LinearAlgebra

using Printf

"""
	read_strd(name)

Read the NIST StRD file `name.dat` in this folder.

Gives: `data`, a matrix with one column per variable, the response first, and `header`, the text
above the data, which holds the certified values.
"""
function read_strd(name)
	lines = readlines(name * ".dat")
	range = match(r"Data\s*:?\s*\(?lines\s+(?<first>\d+)\s+to\s+(?<last>\d+)", join(lines[1:12], "\n"))
	first, last = parse(Int, range[:first]), parse(Int, range[:last])
	data = permutedims(reduce(hcat, [parse.(Float64, split(line)) for line in lines[first:last]]))
	return (data = data, header = join(lines[1:first - 1], "\n"))
end

"""
	certified(header, pattern)

The number captured as `v` by `pattern` in a file's header.
"""
certified(header, pattern) = parse(Float64, match(pattern, header)[:v])

"""
	digits_same(value, certified)

The number of significant digits `value` shares with `certified`: McCullough's log relative
error, or the log absolute error when `certified` is 0, rounded down and at most 15.
"""
function digits_same(value, certified)
	isfinite(value) || return 0
	error = certified == 0 ? abs(value) : abs(value - certified) / abs(certified)
	return error == 0 ? 15 : clamp(floor(Int, -log10(error)), 0, 15)
end

"""
	report(title, columns, rows)

Print each certified value with the values of `columns` below it, and the significant digits
each shares with NIST. Each row is `(label, certified, values...)`, one value for each column.

Gives: the fewest digits shared in each column.
"""
function report(title, columns, rows)
	println("\n", title)
	fewest = fill(15, length(columns))
	for (label, c, values...) in rows
		@printf("%-16s%-12s%25.15g\n", label, "certified", c)
		for (k, (column, v)) in enumerate(zip(columns, values))
			d = digits_same(v, c)
			fewest[k] = min(fewest[k], d)
			@printf("%-16s%-12s%25.15g  %s\n", "", column, v, d == 0 ? "DIFFERENT" : "same to $d significant digits")
		end
	end
	return fewest
end

"""
	summary(columns, results)

Print the fewest significant digits shared with NIST in each dataset, one column for each of
`columns`. Each result is `(name, level, fewest)`.
"""
function summary(columns, results)
	@printf("%-12s%-12s", "dataset", "difficulty")
	foreach(column -> @printf("%14s", column), columns)
	println()
	for (name, level, fewest) in results
		@printf("%-12s%-12s", name, level)
		foreach(d -> @printf("%14s", d == 0 ? "DIFFERENT" : string(d)), fewest)
		println()
	end
end

## 1. The datasets and their models

Each dataset with NIST's difficulty and its model: the degree of the polynomial in `x`, or `nothing` for Longley, whose 6 predictors are the file's columns, and whether the model has an intercept.

In [ ]:
datasets = [
	"Norris" => ("lower", 1, true), "Pontius" => ("lower", 2, true),
	"NoInt1" => ("average", 1, false), "NoInt2" => ("average", 1, false),
	"Filip" => ("higher", 10, true), "Longley" => ("higher", nothing, true),
	"Wampler1" => ("higher", 5, true), "Wampler2" => ("higher", 5, true), "Wampler3" => ("higher", 5, true),
	"Wampler4" => ("higher", 5, true), "Wampler5" => ("higher", 5, true),
]

## 2. Fit by Cholesky and by QR

GLM's `lm` solves the normal equations X'X b = X'y by a Cholesky decomposition, its only method in GLM 1.9.5, the latest release. By default it also drops any predictor it judges collinear with the others; the column `lm, keep all` turns that off with `dropcollinear = false`. Forming X'X squares the condition number of X, so the normal equations lose about twice as many digits as a QR decomposition of X itself when the predictors are nearly collinear, as the powers of `x` in Filip and Wampler are. `qr_fit` is that QR fit, written out with LinearAlgebra's `qr`; the same function on `BigFloat` copies of the data gives the data limit. R-squared is centred for models with an intercept and uncentred for the two without, as NIST defines it.

In [ ]:
"""
	design(data, degree, intercept)

The predictor matrix of a NIST file: the powers 1 to `degree` of the second column, or, when
`degree` is `nothing`, all columns after the first; with a column of ones first if `intercept`.
"""
function design(data, degree, intercept)
	X = degree === nothing ? data[:, 2:end] : data[:, 2] .^ (1:degree)'
	return intercept ? hcat(ones(size(X, 1)), X) : X
end

"""
	glm_fit(X, y; dropcollinear = true)

Fit `y` on `X` with GLM's `lm`; by default it drops a predictor it judges collinear with the
others, giving it a coefficient of 0 and a standard error of `NaN`. When the Cholesky decomposition
fails, every value is `NaN`.

Gives: the coefficients, standard errors, residual standard deviation and R-squared.
"""
function glm_fit(X, y; dropcollinear = true)
	model = try
		lm(X, y; dropcollinear = dropcollinear)
	catch e
		e isa PosDefException || rethrow()
		return (coef = fill(NaN, size(X, 2)), se = fill(NaN, size(X, 2)), sd = NaN, r2 = NaN)
	end
	return (coef = coef(model), se = stderror(model), sd = sqrt(deviance(model) / dof_residual(model)), r2 = r2(model))
end

"""
	qr_fit(X, y, intercept)

The same values from a QR decomposition of `X`, in the number type of `X` and `y`. The standard
errors come from (X'X)^-1 = R^-1 R^-T, whose diagonal is the row sums of squares of R^-1.
"""
function qr_fit(X, y, intercept)
	F = qr(X)
	b = F \ y
	rss = sum(abs2, y - X * b)
	s2 = rss / (size(X, 1) - size(X, 2))
	Rinv = inv(UpperTriangular(F.R))
	total = intercept ? sum(abs2, y .- sum(y) / length(y)) : sum(abs2, y)
	return (coef = b, se = sqrt.(s2 .* vec(sum(abs2, Rinv; dims = 2))), sd = sqrt(s2), r2 = 1 - rss / total)
end

columns = ["lm", "lm, keep all", "QR", "data limit"]
results = []
for (name, (level, degree, intercept)) in datasets
	(; data, header) = read_strd(name)
	X, y = design(data, degree, intercept), data[:, 1]
	fits = (glm_fit(X, y), glm_fit(X, y; dropcollinear = false), qr_fit(X, y, intercept), qr_fit(big.(X), big.(y), intercept))
	table = [parse.(Float64, m.captures) for m in eachmatch(r"\n *B(?<k>\d+) +(?<estimate>\S+) +(?<se>\S+)", header)]
	rows = []
	for (i, (k, estimate, se)) in enumerate(table)
		push!(rows, ("B$(Int(k))", estimate, (f.coef[i] for f in fits)...))
		push!(rows, ("  SE", se, (f.se[i] for f in fits)...))
	end
	push!(rows, ("residual SD", certified(header, r"Standard Deviation +(?<v>\S+)"), (f.sd for f in fits)...))
	push!(rows, ("R-squared", certified(header, r"R-Squared +(?<v>\S+)"), (f.r2 for f in fits)...))
	fewest = report("$name ($level difficulty, $(length(y)) observations)", columns, rows)
	push!(results, (name, level, fewest))
end

## 3. Summary

The fewest significant digits shared with NIST in each dataset: by GLM's `lm` with its defaults and with `dropcollinear = false`, by QR, and at the data limit.

In [ ]:
summary(columns, results)

## What the summary shows

GLM's `lm` with its defaults gives no right value on Pontius, Longley and Filip. It judges some predictors collinear with the others and drops them: the intercept of Pontius and Longley, and the intercept and the first five powers of x in Filip. It then fits a smaller model, the dropped coefficients print as 0, and no warning is given. With `dropcollinear = false` it gives 12 digits on Pontius and 7 on Longley, but its Cholesky decomposition fails on Filip, where X'X is not positive definite in Float64. On the Wampler polynomials it gives 6 digits, except 9 or 10 on Wampler2.

QR gives more digits than the normal equations on Longley and four of the Wampler datasets, and 1 fewer on Pontius and Wampler5. It is the only Float64 method here that fits Filip at all, to 7 digits against a data limit of 8: its powers of x up to x^10 are so nearly collinear that storing the data alone costs 7 of the 15 digits. For polynomials and other nearly collinear predictors, prefer QR (Julia's `X \ y`) to the normal equations, and check that no coefficient was dropped.